In [1]:

from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd
import pickle
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
mpl.rcParams['font.size'] = 10

In [2]:
import time
import sys
import os,glob
from collections import deque
from typing import Dict, Tuple

import gymnasium as gym
import numpy as np
import torch
from torch import Tensor

from sample_factory.algo.learning.learner import Learner
from sample_factory.algo.sampling.batched_sampling import preprocess_actions
from sample_factory.algo.utils.action_distributions import argmax_actions
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.rl_utils import make_dones, prepare_and_normalize_obs
from sample_factory.algo.utils.tensor_utils import unsqueeze_tensor
from sample_factory.cfg.arguments import load_from_checkpoint
# from sample_factory.huggingface.huggingface_utils import generate_model_card, generate_replay_video, push_to_hf
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config, StatusCode
from sample_factory.utils.utils import debug_log_every_n, experiment_dir, log

# ---------------------------------------------------------------------------
# logging helpers (put near the top of the file, after imports)
# ---------------------------------------------------------------------------
import datetime, pathlib, json, pandas as pd, torch

def _ensure_parent(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

/home/fr/fr_lr554/.conda/envs/env/lib/python3.10/site-packages/deepmind_lab/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import sys
from multiprocessing.context import BaseContext
from typing import Optional

from tensorboardX import SummaryWriter

from sample_factory.algo.runners.runner import AlgoObserver, Runner
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.multiprocessing_utils import get_mp_ctx
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import make_runner
from sample_factory.utils.typing import Config, Env, PolicyID
from sample_factory.utils.utils import experiment_dir

# from sf_workingdir_lilly.dmlab.dmlab_env import (
#     DMLAB_ENVS,
#     dmlab_extra_episodic_stats_processing,
#     dmlab_extra_summaries,
#     list_all_levels_for_experiment,
#     make_dmlab_env,
# )
from sf_workingdir_lilly.dmlab.dmlab_level_cache import DmlabLevelCaches, make_dmlab_caches
# from sf_examples.dmlab.dmlab_model import make_dmlab_encoder
from sf_workingdir_lilly.dmlab.custom_core import make_hipposlam_core
from sf_workingdir_lilly.dmlab.custom_encoder import make_hipposlam_encoder
from sf_workingdir_lilly.dmlab.dmlab_params import add_dmlab_env_args, dmlab_override_defaults
from sf_workingdir_lilly.dmlab.custom_params import add_hipposlam_env_args, hipposlam_override_defaults


class DmlabEnvWithCache:
    def __init__(self, level_caches: Optional[DmlabLevelCaches] = None):
        self.caches = level_caches

    def make_env(self, env_name, cfg, env_config, render_mode) -> Env:
        return make_dmlab_env(env_name, cfg, env_config, render_mode, self.caches)


def register_dmlab_envs(level_caches: Optional[DmlabLevelCaches] = None):
    env_factory = DmlabEnvWithCache(level_caches)
    for env in DMLAB_ENVS:
        register_env(env.name, env_factory.make_env)


def register_dmlab_components(level_caches: Optional[DmlabLevelCaches] = None):
    # register_dmlab_envs(level_caches)
    global_model_factory().register_encoder_factory(make_hipposlam_encoder)
    global_model_factory().register_model_core_factory(make_hipposlam_core)


class DmlabExtraSummariesObserver(AlgoObserver):
    def extra_summaries(self, runner: Runner, policy_id: PolicyID, writer: SummaryWriter, env_steps: int) -> None:
        dmlab_extra_summaries(runner, policy_id, writer, env_steps)


def register_msg_handlers(cfg: Config, runner: Runner):
    if cfg.env == "dmlab_30":
        # extra functions to calculate human-normalized score etc.
        runner.register_episodic_stats_handler(dmlab_extra_episodic_stats_processing)
        runner.register_observer(DmlabExtraSummariesObserver())


def initialize_level_cache(cfg: Config, mp_ctx: BaseContext) -> Optional[DmlabLevelCaches]:
    if not cfg.dmlab_use_level_cache:
        return None

    env_name = cfg.env
    num_policies = cfg.num_policies if hasattr(cfg, "num_policies") else 1
    all_levels = list_all_levels_for_experiment(env_name)
    level_cache_dir = cfg.dmlab_level_cache_path
    caches = make_dmlab_caches(experiment_dir(cfg), all_levels, num_policies, level_cache_dir, mp_ctx)
    return caches


def parse_dmlab_args(argv=None, evaluation=False):
    parser, cfg = parse_sf_args(argv, evaluation=evaluation)
    add_hipposlam_env_args(parser)
    add_dmlab_env_args(parser)
    hipposlam_override_defaults(parser)
    cfg = parse_full_cfg(parser, argv)
    return cfg



In [4]:

REDUCED_ACTION_SET = (
    (0, 0, 0, 1, 0, 0, 0),  # Forward
    # (0, 0, 0, -1, 0, 0, 0),  # Backward
    (0, 0, -1, 0, 0, 0, 0),  # Strafe Left
    (0, 0, 1, 0, 0, 0, 0),  # Strafe Right
    # (-20, 0, 0, 0, 0, 0, 0),  # Look Left
    # (20, 0, 0, 0, 0, 0, 0),  # Look Right
    (-20, 0, 0, 1, 0, 0, 0),  # Look Left + Forward
    (20, 0, 0, 1, 0, 0, 0),  # Look Right + Forward
    # (0, 0, 0, 0, 1, 0, 0),  # Fire.
)
action_space = gym.spaces.Discrete(len(REDUCED_ACTION_SET))
observation_space = gym.spaces.Dict(
    obs=gym.spaces.Box(low=0, high=255, shape=[4, 72, 96], dtype=np.uint8)
)

In [5]:
%cd /work/classic/fr_lr554-TrainSpace/spectral_radius

/work/classic/fr_lr554-TrainSpace/spectral_radius


In [6]:
trained_lora = dict()
weight_trials = [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 22, 23, 24, 27, 28, 29, 30, 33, 35, 36, 38, 39, 40, 41, 42, 43, 45, 47, 49]
weight_trials_unordered = [14,39,23,45,8,22,17,15,38,4,11,1,9,3,18,33,6,12,49,43,7,41,35,5,40,42,27,36,13,24,28,30,47,29]
seeds = [1111,2222,3333, 4444,5555]

simulation = 'RNNRandomLORA48'

In [7]:
ids = [f"{i:02d}" for i in range(170)]
print(ids)

['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', 

In [8]:
# the final dataframe will have the structure dict with weight trials, dict with seeds, and then the two lora components in there
all_lora = dict()

In [9]:
for weight_i, t in enumerate(weight_trials_unordered):
    df_lora_current_weight_trial = {1111:{'lr_column':None, 'lr_row':None}, 
                    2222:{'lr_column':None, 'lr_row':None}, 
                    3333:{'lr_column':None, 'lr_row':None},
                    4444:{'lr_column':None, 'lr_row':None},
                    5555:{'lr_column':None, 'lr_row':None}}
    for seed_i, seed in enumerate(seeds):
        cfg_filename=f'/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/{simulation}_/{ids[weight_i+len(weight_trials_unordered)*seed_i]}_{simulation}_see_{seed}_w.tri_{t}/config.json'
        with open(cfg_filename, "r") as json_file:
            json_params = json.load(json_file)
            log.warning("Loading existing experiment configuration from %s", cfg_filename)
            loaded_cfg = AttrDict(json_params)

        cfg=loaded_cfg

        mapname="openfield_map2_fixed_loc3"
        expname=f'hipposlam/{simulation}_/{ids[weight_i+len(weight_trials_unordered)*seed_i]}_{simulation}_see_{seed}_w.tri_{t}'

        cli = [
            "--algo", "APPO",
            "--env", mapname ,         # pick any DM‑Lab level you have
            "--experiment", expname,
            "--encoder_load_path","/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth",
            "--train_dir", "/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir", # anything writable
            "--max_num_frames", "50000",          # short rollout for the test
            "--num_envs", "8",
            "--dmlab_level_cache_path","./.dmlab_cache",
            "--load_checkpoint_kind","latest",
            "--use_jit","False",
            "--with_pos_obs","True",
            "--depth_sensor", "True",
            "--no_render",        # <-- skip human window; avoid X11 on servers
        ]

        cli_dict={
        'algo': 'APPO',
        'env': mapname,
        'experiment': expname,
        'encoder_load_path': '/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth',
        'train_dir': '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir',
        'max_num_frames': '50000',
        'num_envs': '8',
        'dmlab_level_cache_path': './.dmlab_cache',
        'load_checkpoint_kind': 'latest',
        'no_render': True,
        'use_jit': False,
        'with_pos_obs': True,
        "depth_sensor": True,
        }
        register_dmlab_components()
        cfg = parse_dmlab_args(evaluation=True, argv=cli)

        cfg.train_dir = "/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir"
        cfg.experiment = expname

        pattern = os.path.join(cfg.train_dir, cfg.experiment, "**", "*.pth")
        ckpts = glob.glob(pattern, recursive=True)
        assert ckpts, f"No checkpoints found under {pattern}"
        ckpt_path = max(ckpts, key=os.path.getmtime)
        print("Using checkpoint:", ckpt_path)

        # tweak whatever you like *after* parsing
        # cfg.with_pos_obs = True
        cfg.cli_args=cli_dict
        # status = enjoy(cfg)


        verbose = False

        cfg = load_from_checkpoint(cfg)

        eval_env_frameskip: int = cfg.env_frameskip if cfg.eval_env_frameskip is None else cfg.eval_env_frameskip
        assert (
            cfg.env_frameskip % eval_env_frameskip == 0
        ), f"{cfg.env_frameskip=} must be divisible by {eval_env_frameskip=}"
        render_action_repeat: int = cfg.env_frameskip // eval_env_frameskip
        cfg.env_frameskip = cfg.eval_env_frameskip = eval_env_frameskip
        log.debug(f"Using frameskip {cfg.env_frameskip} and {render_action_repeat=} for evaluation")

        cfg.num_envs = 1

        render_mode = "human"
        if cfg.save_video:
            render_mode = "rgb_array"
        elif cfg.no_render:
            render_mode = None

        # env = make_env_func_batched(
        #     cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=render_mode
        # )
        # env_info = extract_env_info(env, cfg)
        # if hasattr(env.unwrapped, "reset_on_init"):
        #     # reset call ruins the demo recording for VizDoom
        #     env.unwrapped.reset_on_init = False
        # log.info(env.action_space)
        actor_critic = create_actor_critic(cfg, observation_space, action_space)


        actor_critic.eval()
        policy_id = cfg.policy_index
        # log.info(policy_id)
        name_prefix = dict(latest="checkpoint", best="best")[cfg.load_checkpoint_kind]
        # log.info(Learner.checkpoint_dir(cfg, policy_id))
        # checkpoints = Learner.get_checkpoints(Learner.checkpoint_dir(cfg, policy_id), f"{name_prefix}_*")
        # checkpoint_dict = Learner.load_checkpoint(str(pth_name), device)
        checkpoint_dict = torch.load(str(ckpt_path), 'cpu', weights_only=False)
        actor_critic.load_state_dict(checkpoint_dict["model"])




        print('#######################################')
        print(t, seed)
        df_lora_current_weight_trial[seed]['lr_column'] =  actor_critic.core.rnn.lr_column.detach().cpu().numpy()
        df_lora_current_weight_trial[seed]['lr_row'] =  actor_critic.core.rnn.lr_row.detach().cpu().numpy()
        print(actor_critic.core.rnn.lr_column)
        print(actor_critic.core.rnn.lr_row)
        #break
    all_lora[t] = df_lora_current_weight_trial
    #break
        

[2026-02-10 10:26:43,915][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/00_RNNRandomLORA48_see_1111_w.tri_14/config.json
[2026-02-10 10:26:43,916][2428772] register_encoder_factory: <function make_hipposlam_encoder at 0x7fa89dd75750>
[2026-02-10 10:26:43,916][2428772] register_model_core_factory: <function make_hipposlam_core at 0x7fa8aa470b80>


[2026-02-10 10:26:43,954][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/00_RNNRandomLORA48_see_1111_w.tri_14/config.json
[2026-02-10 10:26:43,955][2428772] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA48_/00_RNNRandomLORA48_see_1111_w.tri_14' passed from command line
[2026-02-10 10:26:43,955][2428772] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-10 10:26:43,956][2428772] Overriding arg 'dmlab_level_cache_path' with value './.dmlab_cache' passed from command line
[2026-02-10 10:26:43,956][2428772] Overriding arg 'use_jit' with value False passed from command line
[2026-02-10 10:26:43,956][2428772] Overriding arg 'with_pos_obs' with value True passed from command line
[2026-02-10 10:26:43,957][2428772] Adding new argument 'fps'=0 that is not in the saved config file!
[2026-02-10 10:26:43

Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/00_RNNRandomLORA48_see_1111_w.tri_14/checkpoint_p3/checkpoint_000010899_89284608.pth
#######################################
14 1111
Parameter containing:
tensor([[-2.5482e-02, -6.9556e-03, -1.5147e-02,  ...,  5.5466e-03,
          1.6817e-02, -1.8089e-03],
        [ 8.4153e-05, -9.1338e-03, -4.9691e-02,  ...,  1.0916e-02,
          3.9869e-02,  5.3227e-04],
        [ 2.2832e-02,  9.9131e-03, -1.0098e-02,  ..., -2.1479e-02,
         -1.4038e-02,  6.7302e-04],
        ...,
        [-2.6039e-02, -1.0852e-02, -2.8658e-02,  ..., -1.7936e-02,
         -8.4944e-03, -4.2094e-02],
        [-1.2780e-03, -8.5984e-04,  1.1933e-02,  ..., -6.8129e-03,
          1.1556e-02,  2.6941e-02],
        [ 1.6609e-02,  1.0175e-02,  1.1907e-03,  ..., -1.7368e-02,
          3.1899e-03, -1.6935e-02]], requires_grad=True)
Parameter containing:
tensor([[ 0.0198,  0.0092,  0.0127,  ...,  0.0441,  0.0257,  0.018

[2026-02-10 10:26:44,159][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/34_RNNRandomLORA48_see_2222_w.tri_14/config.json
[2026-02-10 10:26:44,160][2428772] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA48_/34_RNNRandomLORA48_see_2222_w.tri_14' passed from command line
[2026-02-10 10:26:44,160][2428772] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-10 10:26:44,160][2428772] Overriding arg 'dmlab_level_cache_path' with value './.dmlab_cache' passed from command line
[2026-02-10 10:26:44,160][2428772] Overriding arg 'use_jit' with value False passed from command line
[2026-02-10 10:26:44,160][2428772] Overriding arg 'with_pos_obs' with value True passed from command line
[2026-02-10 10:26:44,161][2428772] Adding new argument 'fps'=0 that is not in the saved config file!
[2026-02-10 10:26:44

Using checkpoint: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/34_RNNRandomLORA48_see_2222_w.tri_14/checkpoint_p3/checkpoint_000016188_132612096.pth
#######################################
14 2222
Parameter containing:
tensor([[-0.0077,  0.0002, -0.0481,  ..., -0.0012,  0.0187,  0.0169],
        [ 0.0128, -0.0142,  0.0068,  ..., -0.0201, -0.0206, -0.0139],
        [-0.0075, -0.0256, -0.0259,  ..., -0.0025,  0.0126, -0.0336],
        ...,
        [ 0.0031,  0.0008,  0.0277,  ...,  0.0331,  0.0132, -0.0229],
        [-0.0073,  0.0357,  0.0028,  ..., -0.0041,  0.0384,  0.0015],
        [-0.0111,  0.0263, -0.0047,  ...,  0.0154,  0.0330,  0.0094]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0140,  0.0036, -0.0007,  ..., -0.0203,  0.0296, -0.0066],
        [-0.0018,  0.0007,  0.0253,  ...,  0.0325, -0.0079,  0.0243],
        [ 0.0094,  0.0331,  0.0141,  ...,  0.0041,  0.0006,  0.0173],
        ...,
        [-0.0229,  0.0251, -0.015

...

#######################################
47 4444
Parameter containing:
tensor([[-0.0129,  0.0221,  0.0133,  ...,  0.0140,  0.0197, -0.0266],
        [-0.0293,  0.0079, -0.0187,  ..., -0.0259,  0.0244,  0.0060],
        [-0.0268, -0.0010,  0.0213,  ..., -0.0131,  0.0208,  0.0252],
        ...,
        [-0.0289,  0.0039,  0.0219,  ..., -0.0453, -0.0228, -0.0074],
        [-0.0031, -0.0041, -0.0325,  ..., -0.0023, -0.0224,  0.0211],
        [-0.0187,  0.0270,  0.0144,  ..., -0.0012,  0.0124, -0.0106]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0154,  0.0197, -0.0235,  ..., -0.0434,  0.0249, -0.0111],
        [-0.0030, -0.0277, -0.0086,  ..., -0.0443,  0.0058, -0.0223],
        [ 0.0393,  0.0342,  0.0091,  ...,  0.0582,  0.0229, -0.0226],
        ...,
        [-0.0095, -0.0011, -0.0083,  ...,  0.0282,  0.0010, -0.0315],
        [ 0.0181, -0.0218,  0.0194,  ..., -0.0093,  0.0069, -0.0314],
        [ 0.0206,  0.0008,  0.0138,  ..., -0.0006,  0.0073,  0.0045]],
       requir

[2026-02-10 10:27:02,651][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/67_RNNRandomLORA48_see_2222_w.tri_29/config.json
[2026-02-10 10:27:02,652][2428772] register_encoder_factory: <function make_hipposlam_encoder at 0x7fa89dd75750>
[2026-02-10 10:27:02,652][2428772] register_model_core_factory: <function make_hipposlam_core at 0x7fa8aa470b80>
[2026-02-10 10:27:02,675][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/67_RNNRandomLORA48_see_2222_w.tri_29/config.json
[2026-02-10 10:27:02,676][2428772] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA48_/67_RNNRandomLORA48_see_2222_w.tri_29' passed from command line
[2026-02-10 10:27:02,676][2428772] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-10 1

#######################################
29 1111
Parameter containing:
tensor([[-0.0281, -0.0153, -0.0139,  ...,  0.0126,  0.0177, -0.0095],
        [ 0.0225,  0.0169, -0.0110,  ..., -0.0097,  0.0193,  0.0062],
        [ 0.0410,  0.0128, -0.0172,  ..., -0.0392, -0.0102, -0.0169],
        ...,
        [-0.0148,  0.0041,  0.0042,  ..., -0.0120, -0.0129, -0.0272],
        [ 0.0014,  0.0011, -0.0065,  ..., -0.0206,  0.0016,  0.0132],
        [ 0.0129, -0.0163, -0.0128,  ..., -0.0181,  0.0145, -0.0161]],
       requires_grad=True)
Parameter containing:
tensor([[ 0.0346,  0.0321,  0.0199,  ...,  0.0402,  0.0226,  0.0353],
        [-0.0093,  0.0081,  0.0203,  ..., -0.0051,  0.0072,  0.0397],
        [ 0.0037, -0.0120,  0.0180,  ...,  0.0098,  0.0025,  0.0152],
        ...,
        [ 0.0238, -0.0095, -0.0268,  ...,  0.0201,  0.0039,  0.0117],
        [ 0.0036,  0.0221,  0.0197,  ..., -0.0115,  0.0057, -0.0202],
        [ 0.0233, -0.0393,  0.0092,  ..., -0.0165, -0.0077, -0.0276]],
       requir

[2026-02-10 10:27:02,875][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/135_RNNRandomLORA48_see_4444_w.tri_29/config.json
[2026-02-10 10:27:02,876][2428772] register_encoder_factory: <function make_hipposlam_encoder at 0x7fa89dd75750>
[2026-02-10 10:27:02,876][2428772] register_model_core_factory: <function make_hipposlam_core at 0x7fa8aa470b80>
[2026-02-10 10:27:02,904][2428772] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/135_RNNRandomLORA48_see_4444_w.tri_29/config.json
[2026-02-10 10:27:02,905][2428772] Overriding arg 'experiment' with value 'hipposlam/RNNRandomLORA48_/135_RNNRandomLORA48_see_4444_w.tri_29' passed from command line
[2026-02-10 10:27:02,905][2428772] Overriding arg 'train_dir' with value '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir' passed from command line
[2026-02-1

#######################################
29 3333
Parameter containing:
tensor([[-0.0173,  0.0134, -0.0212,  ...,  0.0097, -0.0112, -0.0156],
        [-0.0402,  0.0059,  0.0146,  ..., -0.0081, -0.0087, -0.0016],
        [ 0.0040,  0.0025, -0.0122,  ...,  0.0389,  0.0402, -0.0422],
        ...,
        [-0.0085, -0.0337, -0.0084,  ...,  0.0144,  0.0396, -0.0128],
        [-0.0019,  0.0200, -0.0078,  ...,  0.0015, -0.0149,  0.0006],
        [-0.0244,  0.0356,  0.0235,  ..., -0.0160, -0.0124, -0.0382]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0330, -0.0037,  0.0041,  ...,  0.0335, -0.0020, -0.0271],
        [ 0.0255,  0.0128, -0.0326,  ...,  0.0093, -0.0105, -0.0097],
        [-0.0082, -0.0240,  0.0221,  ..., -0.0465, -0.0400,  0.0025],
        ...,
        [ 0.0162,  0.0117, -0.0283,  ...,  0.0112, -0.0097, -0.0136],
        [-0.0291, -0.0014, -0.0334,  ...,  0.0084,  0.0074, -0.0247],
        [ 0.0154,  0.0170, -0.0113,  ..., -0.0237, -0.0085, -0.0453]],
       requir

In [10]:
cfg.pixel_format


'CHW'

In [11]:
ckpt_path

'/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29/checkpoint_p0/checkpoint_000000000_0.pth'

In [12]:
print(all_lora.keys())
print(len(all_lora.keys()))
print(all_lora[41][1111])

dict_keys([14, 39, 23, 45, 8, 22, 17, 15, 38, 4, 11, 1, 9, 3, 18, 33, 6, 12, 49, 43, 7, 41, 35, 5, 40, 42, 27, 36, 13, 24, 28, 30, 47, 29])
34
{'lr_column': array([[-0.0328615 ,  0.00319085, -0.03198854, ..., -0.0052863 ,
         0.02184644, -0.00708502],
       [-0.00067304, -0.01462894, -0.00448666, ..., -0.00728466,
         0.01124419, -0.00100956],
       [ 0.0067995 ,  0.01333642,  0.02007095, ...,  0.00369537,
        -0.03383127, -0.0042388 ],
       ...,
       [-0.03642549,  0.03097365,  0.03357954, ..., -0.03066929,
        -0.02172444, -0.04537217],
       [-0.02537229, -0.02436918,  0.02931922, ...,  0.00725755,
         0.00119493,  0.02940309],
       [ 0.04277612, -0.00593931, -0.01562881, ..., -0.03385963,
         0.03277792, -0.02949479]], shape=(1136, 16), dtype=float32), 'lr_row': array([[ 0.01697609,  0.01881848,  0.02402609, ...,  0.03193601,
         0.01310255,  0.00666521],
       [ 0.01312669,  0.02212233,  0.01745735, ...,  0.00020313,
         0.01203807, 

In [13]:
print("train_dir:", cfg.train_dir)
print("experiment:", cfg.experiment)


train_dir: /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir
experiment: hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29


In [14]:
import glob, os

pattern = os.path.join(cfg.train_dir, cfg.experiment, "**", "*.pth")
ckpts = glob.glob(pattern, recursive=True)
print("Found ckpts:", ckpts[:5], " ... total:", len(ckpts))
assert len(ckpts) > 0, "No checkpoint files found for this experiment!"


Found ckpts: ['/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29/checkpoint_p2/checkpoint_000000000_0.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29/checkpoint_p0/checkpoint_000000000_0.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29/checkpoint_p1/checkpoint_000000000_0.pth', '/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNRandomLORA48_/169_RNNRandomLORA48_see_5555_w.tri_29/checkpoint_p3/checkpoint_000000000_0.pth']  ... total: 4


In [15]:
trained_lora[5555]= {'lr_column': actor_critic.core.rnn.lr_column.detach().cpu().numpy(),
                    'lr_row': actor_critic.core.rnn.lr_row.detach().cpu().numpy()}

In [16]:
#print(trained_lora[5555])
print(all_lora[41][1111]['lr_column'])

[[-0.0328615   0.00319085 -0.03198854 ... -0.0052863   0.02184644
  -0.00708502]
 [-0.00067304 -0.01462894 -0.00448666 ... -0.00728466  0.01124419
  -0.00100956]
 [ 0.0067995   0.01333642  0.02007095 ...  0.00369537 -0.03383127
  -0.0042388 ]
 ...
 [-0.03642549  0.03097365  0.03357954 ... -0.03066929 -0.02172444
  -0.04537217]
 [-0.02537229 -0.02436918  0.02931922 ...  0.00725755  0.00119493
   0.02940309]
 [ 0.04277612 -0.00593931 -0.01562881 ... -0.03385963  0.03277792
  -0.02949479]]


In [17]:
# save the extracted data
with open("/home/fr/fr_lr554/samplefactory/sample-factory/sf_workingdir_lilly/dmlab/analysis/data/sim48_lora.pkl", "wb") as f:
    pickle.dump(all_lora, f)

In [ ]:
########################## STOP ##########################

In [ ]:
import torch.nn as nn

In [ ]:
print(type(actor_critic.core))
print(isinstance(actor_critic.core, nn.Module))

print("children:", list(actor_critic.core.named_children()))
print("modules :", list(actor_critic.core.named_modules())[:10])
print("params  :", list(actor_critic.core.named_parameters())[:10])

<class 'sf_examples.dmlab.Hipposlam_model.SimpleSequenceWithBypassCore'>
True
children: []
modules : [('', SimpleSequenceWithBypassCore())]
params  : []


In [ ]:
actor_critic.core.named_parameters()

dict_keys([])

In [ ]:
dict_keys(['', 'obs_normalizer', 'obs_normalizer.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std.obs', 'returns_normalizer', 'encoder', 'encoder.depth_encoder', 'encoder.depth_encoder.downsample', 'encoder.basic_encoder', 'encoder.basic_encoder.conv_head', 'encoder.basic_encoder.conv_head.0', 'encoder.basic_encoder.conv_head.1', 'encoder.basic_encoder.conv_head.2', 'encoder.basic_encoder.conv_head.2.res_block_core', 'encoder.basic_encoder.conv_head.2.res_block_core.0', 'encoder.basic_encoder.conv_head.2.res_block_core.1', 'encoder.basic_encoder.conv_head.2.res_block_core.2', 'encoder.basic_encoder.conv_head.2.res_block_core.3', 'encoder.basic_encoder.conv_head.3', 'encoder.basic_encoder.conv_head.3.res_block_core', 'encoder.basic_encoder.conv_head.3.res_block_core.0', 'encoder.basic_encoder.conv_head.3.res_block_core.1', 'encoder.basic_encoder.conv_head.3.res_block_core.2', 'encoder.basic_encoder.conv_head.3.res_block_core.3', 'encoder.basic_encoder.conv_head.4', 'encoder.basic_encoder.conv_head.5', 'encoder.basic_encoder.conv_head.6', 'encoder.basic_encoder.conv_head.6.res_block_core', 'encoder.basic_encoder.conv_head.6.res_block_core.0', 'encoder.basic_encoder.conv_head.6.res_block_core.1', 'encoder.basic_encoder.conv_head.6.res_block_core.2', 'encoder.basic_encoder.conv_head.6.res_block_core.3', 'encoder.basic_encoder.conv_head.7', 'encoder.basic_encoder.conv_head.7.res_block_core', 'encoder.basic_encoder.conv_head.7.res_block_core.0', 'encoder.basic_encoder.conv_head.7.res_block_core.1', 'encoder.basic_encoder.conv_head.7.res_block_core.2', 'encoder.basic_encoder.conv_head.7.res_block_core.3', 'encoder.basic_encoder.conv_head.8', 'encoder.basic_encoder.conv_head.9', 'encoder.basic_encoder.conv_head.10', 'encoder.basic_encoder.conv_head.10.res_block_core', 'encoder.basic_encoder.conv_head.10.res_block_core.0', 'encoder.basic_encoder.conv_head.10.res_block_core.1', 'encoder.basic_encoder.conv_head.10.res_block_core.2', 'encoder.basic_encoder.conv_head.10.res_block_core.3', 'encoder.basic_encoder.conv_head.11', 'encoder.basic_encoder.conv_head.11.res_block_core', 'encoder.basic_encoder.conv_head.11.res_block_core.0', 'encoder.basic_encoder.conv_head.11.res_block_core.1', 'encoder.basic_encoder.conv_head.11.res_block_core.2', 'encoder.basic_encoder.conv_head.11.res_block_core.3', 'encoder.basic_encoder.conv_head.12', 'encoder.basic_encoder.mlp_layers', 'encoder.basic_encoder.mlp_layers.0', 'encoder.DG_projection', 'encoder.DG_projection.linear', 'encoder.DG_projection.batchnorm1d', 'encoder.DG_projection.activation', 'core', 'decoder', 'decoder.mlp', 'decoder.mlp.0', 'decoder.mlp.1', 'decoder.mlp.2', 'critic_linear', 'action_parameterization', 'action_parameterization.distribution_linear'])

In [ ]:
[x for x in dict(actor_critic.named_modules()).keys() if x.startswith('encoder.DG')]